# Data Exploration — Infrastructure Defect Dataset

This notebook explores aerial inspection images and defect annotations.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.processing.image_processor import ImageProcessor

DATA_DIR = Path('../data/raw')
processor = ImageProcessor()

In [ ]:
# List available images
images = list(DATA_DIR.glob('*.jpg')) + list(DATA_DIR.glob('*.png'))
print(f'Found {len(images)} images')

if images:
    sample = processor.load(images[0])
    enhanced = processor.enhance(sample)
    edges = processor.extract_edges(sample)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original')
    axes[1].imshow(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB)); axes[1].set_title('Enhanced')
    axes[2].imshow(edges, cmap='gray'); axes[2].set_title('Edge Detection')
    plt.tight_layout(); plt.show()
else:
    print('No images found. Add sample images to data/raw/')

In [ ]:
# Analyze image statistics across the dataset
stats = []
for img_path in images[:50]:
    img = cv2.imread(str(img_path))
    if img is not None:
        stats.append({
            'file': img_path.name,
            'width': img.shape[1],
            'height': img.shape[0],
            'mean_brightness': np.mean(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)),
        })

df = pd.DataFrame(stats)
if not df.empty:
    display(df.describe())
    sns.histplot(df['mean_brightness'], bins=20)
    plt.title('Brightness Distribution'); plt.show()